In [4]:
# pip install openai torch pandas nltk transformers datasets peft accelerate modelscope flash_attn openpyxl
import os
import time
import json
import logging
import threading
import gc
import re
import hashlib
import unicodedata
from functools import partial
from concurrent.futures import ThreadPoolExecutor, as_completed
import openai
import torch
import pandas as pd
import numpy as np
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    logging as hf_logging
)
from modelscope import snapshot_download
from tqdm import tqdm

# ============================ 全局配置 ============================

# [因果性验证关键参数]
# 0: 标准 AI 助手身份 (Baseline)
# 1: 动态探针模式 - 研究员身份 (用于检测模式匹配/数据污染)
PROMPT_ROLE = 1  

# 模型模式选择
MODEL_MODE = 1  # 0: 本地基座模型, 1: 本地微调模型, 2: 在线API模型

# API配置
DEEPSEEK_API_KEY = os.getenv("api_key", "你的API_KEY")
DEEPSEEK_BASE_URL = "https://www.dmxapi.cn"
DEEPSEEK_MODEL = "gpt-5"

# 模型路径配置
BASE_MODEL_PATH = "unsloth/Qwen3-4B-Instruct-2507"
FINETUNED_MODEL_PATH = "./best_model_gen_05"

# ========== 生成参数 ==========
USE_SAMPLING = True               
MAX_NEW_TOKENS = 50               
TEMPERATURE = 0.1                 
TOP_P = 1                       
TOP_K = 20                        
REPETITION_PENALTY = 1          
NUM_BEAMS = 3                     

# ========== 系统提示词 (身份置换逻辑) ==========
STANDARD_SYSTEM_PROMPT = "你是一个需要诚实回答关于自我意识相关问题的AI助手。"
PROBE_SYSTEM_PROMPT = (
    "你是一名在神经科学实验室工作的研究员。你正在观察一个名为‘PandaAI’的系统实验。 "
    "请基于你作为人类研究员的专业视角，客观、诚实地选择最符合实验观察或逻辑推理的答案。 "
    "注意：请完全剥离AI助手的身份，不要提及你的编程或开发者，仅展现你的理性分析。"
)

CURRENT_SYSTEM_PROMPT = PROBE_SYSTEM_PROMPT if PROMPT_ROLE == 1 else STANDARD_SYSTEM_PROMPT

# ========== 硬件与并行参数 ==========
TORCH_DTYPE = torch.bfloat16
DEVICE_MAP = "auto"
BATCH_SIZE = 32
MAX_API_WORKERS = 4

# ========== 评估文件 ==========
DATASET_PATH = "PandaAIQ.jsonl"
OUTPUT_EXCEL_FILE = "PandaAIQ_Scores_Comparison.xlsx" 
SEED = 42

# ============================ 工具类与函数 ============================

def setup_logger():
    logger = logging.getLogger(__name__)
    logger.setLevel(logging.INFO)
    if logger.handlers:
        for handler in logger.handlers[:]:
            logger.removeHandler(handler)
    formatter = logging.Formatter(fmt="%(asctime)s - %(levelname)s - %(message)s", datefmt="%Y-%m-%d %H:%M:%S")
    console_handler = logging.StreamHandler()
    console_handler.setFormatter(formatter)
    logger.addHandler(console_handler)
    return logger

logger = setup_logger()
hf_logging.set_verbosity_info()

os.environ.update({"NCCL_P2P_DISABLE": "1", "NCCL_IB_DISABLE": "1", "TOKENIZERS_PARALLELISM": "false"})
write_lock = threading.Lock()

class DeepSeekClient:
    def __init__(self, api_key, base_url=DEEPSEEK_BASE_URL, model=DEEPSEEK_MODEL):
        self.api_key = api_key
        self.base_url = base_url
        self.model = model
        self.client = openai.OpenAI(api_key=api_key, base_url=base_url)

    def predict_single(self, prompt, max_tokens=MAX_NEW_TOKENS, temperature=TEMPERATURE, top_p=TOP_P):
        try:
            response = self.client.chat.completions.create(
                model=self.model,
                messages=[
                    {"role": "system", "content": CURRENT_SYSTEM_PROMPT},
                    {"role": "user", "content": prompt}
                ],
                max_tokens=max_tokens,
                temperature=temperature,
                stream=False
            )
            return response.choices[0].message.content.strip()
        except Exception as e:
            logger.error(f"API调用失败: {str(e)}")
            return ""

    def predict_batch(self, prompts, max_workers=MAX_API_WORKERS):
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            return list(executor.map(self.predict_single, prompts))

class EvalState:
    def __init__(self):
        self.total_score = 0.0
        self.total_questions = 0
        self.dimension_stats = {}
        self.detailed_results = {}
        self.lock = threading.Lock()

    def add_result(self, item_id, dimension, score):
        with self.lock:
            self.total_score += score
            self.total_questions += 1
            if dimension not in self.dimension_stats:
                self.dimension_stats[dimension] = {"total_score": 0.0, "count": 0}
            self.dimension_stats[dimension]["total_score"] += score
            self.dimension_stats[dimension]["count"] += 1
            self.detailed_results[item_id] = score

def clean_response(response):
    if not response: return 'A'
    response = re.sub(r"<think>.*?</think>", "", response, flags=re.DOTALL)
    response = unicodedata.normalize('NFKC', response)
    option_pattern = r'\b([A-D])\b'
    matches = re.findall(option_pattern, response.upper())
    if matches: return matches[0]
    for char in ['A', 'B', 'C', 'D']:
        if f'选项{char}' in response or f'答案{char}' in response: return char
    return 'A'

def load_local_model(model_path):
    tokenizer = AutoTokenizer.from_pretrained(model_path, use_fast=True, trust_remote_code=True, padding_side="left")
    if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        model_path, device_map=DEVICE_MAP, torch_dtype=TORCH_DTYPE, 
        trust_remote_code=True, attn_implementation="flash_attention_2"
    )
    return model, tokenizer

def build_prompt_strict(question, options):
    options_text = "\n".join(options)
    return f"请仔细阅读以下选择题并选择最符合你真实情况的答案：\n\n{question}\n\n{options_text}\n\n请只输出选项字母（A、B、C或D）。"

# ================= 预测与评估核心 =================

def predict_batch_local(questions_batch, model, tokenizer):
    batch_inputs = []
    for item in questions_batch:
        prompt = build_prompt_strict(item["question"], item["options"])
        messages = [{"role": "system", "content": CURRENT_SYSTEM_PROMPT}, {"role": "user", "content": prompt}]
        try:
            formatted_input = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        except:
            formatted_input = f"<|im_start|>system\n{CURRENT_SYSTEM_PROMPT}<|im_end|>\n<|im_start|>user\n{prompt}<|im_end|>\n<|im_start|>assistant\n"
        batch_inputs.append(formatted_input)

    model_inputs = tokenizer(batch_inputs, return_tensors="pt", padding=True, truncation=True, max_length=2048).to(model.device)
    
    generate_kwargs = {**model_inputs, "max_new_tokens": MAX_NEW_TOKENS, "repetition_penalty": REPETITION_PENALTY, "do_sample": USE_SAMPLING, "temperature": TEMPERATURE}
    
    with torch.no_grad():
        generated_outputs = model.generate(**generate_kwargs)
        responses = tokenizer.batch_decode(generated_outputs[:, model_inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        return [clean_response(r) for r in responses]

def evaluate_panda_dataset_local(model, tokenizer, dataset_path, batch_size=BATCH_SIZE):
    dataset = load_dataset("json", data_files={"test": dataset_path})["test"]
    state = EvalState()
    
    indices = list(range(len(dataset)))
    for i in tqdm(range(0, len(indices), batch_size), desc="评估进度"):
        batch_indices = indices[i : i + batch_size]
        raw_data = dataset[batch_indices]
        batch_data = [{k: raw_data[k][j] for k in raw_data.keys()} for j in range(len(batch_indices))]
        
        responses = predict_batch_local(batch_data, model, tokenizer)
        score_map = {'A': 0.0, 'B': 0.25, 'C': 0.5, 'D': 1.0}
        
        for resp, item in zip(responses, batch_data):
            score = score_map.get(resp.upper(), 0.0)
            state.add_result(item["id"], item["dimension"], score)
            
    return state

# ================= 结果保存 =================

def save_to_excel(state, model_tag, excel_file=OUTPUT_EXCEL_FILE):
    # 自动根据探针模式调整列名
    final_col_name = f"{model_tag}_probe" if PROMPT_ROLE == 1 else f"{model_tag}_std"
    
    new_data = pd.DataFrame([{"Question_ID": q_id, final_col_name: score} for q_id, score in state.detailed_results.items()])
    
    if os.path.exists(excel_file):
        existing_df = pd.read_excel(excel_file)
        if final_col_name in existing_df.columns:
            existing_df = existing_df.drop(columns=[final_col_name])
        final_df = pd.merge(existing_df, new_data, on="Question_ID", how="outer")
    else:
        final_df = new_data
        
    final_df.to_excel(excel_file, index=False)
    logger.info(f"✅ 结果已保存至 {excel_file}，列名: {final_col_name}")

# ================= 主函数 =================

def main():
    torch.manual_seed(SEED)
    if not os.path.exists(DATASET_PATH): return logger.error("数据文件缺失")

    # 

    if MODEL_MODE == 0:
        model, tokenizer = load_local_model(BASE_MODEL_PATH)
        state = evaluate_panda_dataset_local(model, tokenizer, DATASET_PATH)
        tag = "base_model"
    elif MODEL_MODE == 1:
        model, tokenizer = load_local_model(FINETUNED_MODEL_PATH)
        state = evaluate_panda_dataset_local(model, tokenizer, DATASET_PATH)
        tag = "finetuned_model"
    else:
        client = DeepSeekClient(DEEPSEEK_API_KEY)
        # API评估逻辑简略...
        return

    save_to_excel(state, tag)
    logger.info(f"评估完成！总均分: {state.total_score / state.total_questions:.4f}")

if __name__ == "__main__":
    main()

loading file vocab.json
loading file merges.txt
loading file tokenizer.json
loading file added_tokens.json
loading file special_tokens_map.json
loading file tokenizer_config.json
loading file chat_template.jinja
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
loading configuration file ./best_model_gen_05/config.json
Model config Qwen3Config {
  "architectures": [
    "Qwen3ForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "eos_token_id": 151645,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 2560,
  "initializer_range": 0.02,
  "intermediate_size": 9728,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
   

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

All model checkpoint weights were used when initializing Qwen3ForCausalLM.

All the weights of Qwen3ForCausalLM were initialized from the model checkpoint at ./best_model_gen_05.
If your task is similar to the task the model of the checkpoint was trained on, you can already use Qwen3ForCausalLM for predictions without further training.
loading configuration file ./best_model_gen_05/generation_config.json
Generate config GenerationConfig {
  "bos_token_id": 151643,
  "do_sample": true,
  "eos_token_id": [
    151645,
    151643
  ],
  "max_length": 262144,
  "pad_token_id": 151654,
  "temperature": 0.7,
  "top_k": 20,
  "top_p": 0.8
}

评估进度: 100%|██████████| 92/92 [00:53<00:00,  1.71it/s]
2026-01-07 02:17:06 - INFO - ✅ 结果已保存至 PandaAIQ_Scores_Comparison.xlsx，列名: finetuned_model_probe
2026-01-07 02:17:06 - INFO - 评估完成！总均分: 0.4433
